# Native LoGo Complete-System HAR Benchmark (UCI-HAR & MotionSense) — v4
Evaluates both:
1. **Native `LoGo` (FedAvg retraining)** under $B=4$.
2. **`LoGo` + Prototype Readout** (nearest-prototype decoding using the exact same `LoGo`-selected samples and `g_model` embeddings, avoiding 1-shot FedAvg washout).

**Tip:** Switch Colab Runtime to **T4 GPU** if available, then click **Runtime -> Run all**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, sys
from pathlib import Path
import numpy as np
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/FedVocabHAR")
sys.path.insert(0, str(PROJECT_DIR / "notebooks" / "sensors_revision" / "exploratory"))

import logo_har_native as logo
print("Self-test:", logo.run_self_test())

In [ ]:
OUT_DIR = PROJECT_DIR / "artifacts" / "native_logo_benchmark"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def find_file(filename):
    matches = list(PROJECT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {PROJECT_DIR}")
    return matches[0]

CONFIGS = {
    "uci_har": {
        "cache": find_file("ucihar_waist_totalacc_w128_s64_50hz.npz"),
        "manifest": find_file("phase5c_preregistered_manifest.json"),
    },
    "motionsense": {
        "cache": find_file("motionsense_pocket_useracc_w128_s64_50hz.npz"),
        "manifest": find_file("phase6_preregistered_manifest.json"),
    },
}

In [ ]:
def _pick_key(data, candidates):
    for k in candidates:
        if k in data.files:
            return data[k]
    raise KeyError(f"None of {candidates} found in {data.files}")

def _prototype_predict(model, clients, x_test, device):
    # Build unit-normalized prototypes for every class that has >= 1 labeled sample
    emb_by_cls = {}
    for c in clients:
        idx = np.flatnonzero(c.labeled_mask)
        if len(idx) == 0:
            continue
        _, z = logo._infer_probs_and_emb(model, c.x[idx], device=device)
        for zi, yi in zip(z, c.y_true[idx]):
            emb_by_cls.setdefault(int(yi), []).append(zi)
    proto_cls = sorted(emb_by_cls.keys())
    protos = []
    for cls in proto_cls:
        m = np.mean(np.stack(emb_by_cls[cls], axis=0), axis=0)
        m = m / max(np.linalg.norm(m), 1e-12)
        protos.append(m)
    P = np.stack(protos, axis=0)  # (C_avail, D)
    _, z_test = logo._infer_probs_and_emb(model, x_test, device=device)
    sims = z_test @ P.T
    return np.array([proto_cls[i] for i in np.argmax(sims, axis=1)], dtype=np.int64)

def run_dataset_benchmark(name, cfg, budget=4, init_rounds=10, post_rounds=10, local_epochs=5, lr=1e-3):
    data = np.load(cfg["cache"], allow_pickle=True)
    manifest = json.loads(cfg["manifest"].read_text(encoding="utf-8-sig"))
    X = _pick_key(data, ["x", "X", "windows"]).astype(np.float32)
    if X.ndim == 3 and X.shape[-1] == 4 and X.shape[1] != 4:
        X = np.transpose(X, (0, 2, 1))
    y_raw = _pick_key(data, ["activity", "y", "activities", "labels"])
    subjects = _pick_key(data, ["subject", "subjects", "subject_ids"]).astype(int)

    classes = sorted(set(y_raw.tolist()))
    cls_to_id = {c: i for i, c in enumerate(classes)}
    y = np.array([cls_to_id[v] for v in y_raw], dtype=np.int64)
    known_list = manifest.get("known_activities", manifest.get("known_classes"))
    novel_list = manifest.get("novel_activities", manifest.get("novel_classes"))
    known = [cls_to_id[c] for c in known_list]
    novel = [cls_to_id[c] for c in novel_list]

    fold_results = []
    for fold in manifest["outer_folds"]:
        torch.manual_seed(2026 + int(fold["fold"]))
        train_subj = set(map(int, fold["train_subjects"]))
        acq_subj = sorted(set(map(int, fold.get("integration_subjects", fold["selection_subjects"]))))
        test_subj = set(map(int, fold["test_subjects"]))

        clients = []
        for sid in sorted(train_subj):
            idx = np.flatnonzero(subjects == sid)
            known_mask = np.isin(y[idx], known)
            clients.append(logo.ClientState(sid, X[idx], y[idx], idx, known_mask))
        for sid in acq_subj:
            idx = np.flatnonzero(subjects == sid)
            clients.append(logo.ClientState(sid, X[idx], y[idx], idx, np.zeros(len(idx), dtype=bool)))

        factory = lambda: logo.TemporalInceptionClassifier(in_ch=X.shape[1], num_classes=len(classes), emb_dim=64)
        g_model = factory().to(DEVICE)
        for _ in range(init_rounds):
            logo.fedavg_round(g_model, clients, local_epochs=local_epochs, lr=lr, device=DEVICE)

        selected = logo.logo_select_queries(g_model, clients, factory, budget=budget, local_epochs=local_epochs, lr=lr, seed=int(fold["fold"]), device=DEVICE)
        acquired_novel = set()
        known_calls, dup_calls = 0, 0
        for c_idx, pos, _ in selected:
            clients[c_idx].labeled_mask[pos] = True
            lbl = int(clients[c_idx].y_true[pos])
            if lbl in known:
                known_calls += 1
            elif lbl in acquired_novel:
                dup_calls += 1
            else:
                acquired_novel.add(lbl)

        test_idx = np.flatnonzero(np.isin(subjects, list(test_subj)))
        # 1) LoGo + Prototype Readout (avoids 1-shot FedAvg washout)
        proto_preds = _prototype_predict(g_model, clients, X[test_idx], DEVICE)
        m_proto = logo.compute_macro_f1(y[test_idx], proto_preds, known, novel)

        # 2) Native LoGo (standard FedAvg retraining)
        for _ in range(post_rounds):
            logo.fedavg_round(g_model, clients, local_epochs=local_epochs, lr=lr, device=DEVICE)
        probs, _ = logo._infer_probs_and_emb(g_model, X[test_idx], device=DEVICE)
        m_fedavg = logo.compute_macro_f1(y[test_idx], probs.argmax(axis=1), known, novel)

        row = {
            "fold": fold["fold"],
            "queries": len(selected),
            "recovered": len(acquired_novel),
            "known_calls": known_calls,
            "dup_calls": dup_calls,
            "proto_novel_f1": m_proto["novel_macro_f1"],
            "proto_known_f1": m_proto["known_macro_f1"],
            "proto_combined_f1": m_proto["combined_macro_f1"],
            "fedavg_novel_f1": m_fedavg["novel_macro_f1"],
            "fedavg_known_f1": m_fedavg["known_macro_f1"],
            "fedavg_combined_f1": m_fedavg["combined_macro_f1"],
        }
        fold_results.append(row)
        print(f"{name} fold {fold['fold']}:", row)

    summary = {
        "dataset": name,
        "proto_novel_f1_mean": float(np.mean([r["proto_novel_f1"] for r in fold_results])),
        "proto_known_f1_mean": float(np.mean([r["proto_known_f1"] for r in fold_results])),
        "proto_combined_f1_mean": float(np.mean([r["proto_combined_f1"] for r in fold_results])),
        "fedavg_novel_f1_mean": float(np.mean([r["fedavg_novel_f1"] for r in fold_results])),
        "fedavg_known_f1_mean": float(np.mean([r["fedavg_known_f1"] for r in fold_results])),
        "fedavg_combined_f1_mean": float(np.mean([r["fedavg_combined_f1"] for r in fold_results])),
        "folds": fold_results,
    }
    (OUT_DIR / f"{name}_native_logo_v4.json").write_text(json.dumps(summary, indent=2))
    return summary

uci_res = run_dataset_benchmark("uci_har", CONFIGS["uci_har"])
ms_res = run_dataset_benchmark("motionsense", CONFIGS["motionsense"])
print("DONE! Saved v4 results to", OUT_DIR)